# Table 5 주모형 결과표 생성 노트북

## 1. 목적

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로, 9주차 중간보고서 4. 예비분석 결과 섹션에 들어갈 주모형 결과표를 생성한다.

이번 단계에서는 주모형만 추정한다. 대안모형, 강건성 분석, 보조모형, Ordered Logit, Poisson, Negative Binomial, 보조 DV, 보조 IV 분석은 포함하지 않는다.

## 2. 설정값 및 저장 옵션

`SAVE_OUTPUTS` 플래그 하나로 결과 파일 저장 여부를 관리한다.

- `SAVE_OUTPUTS = True`: 결과표 CSV/Markdown과 해석 메모를 `output/tables`에 저장한다.
- `SAVE_OUTPUTS = False`: 파일 저장 없이 노트북 화면에만 결과를 표시한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

try:
    from IPython.display import display
except ImportError:
    display = None

# 저장 여부 단일 플래그
SAVE_OUTPUTS = False

CWD = Path.cwd()
if (CWD / 'working').exists():
    BASE_DIR = CWD
elif (CWD.parent / 'working').exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError('working 폴더가 있는 프로젝트 루트를 찾지 못했습니다.')

ANALYSIS_DIR = BASE_DIR / 'working' / 'analysis'
OUT_DIR = BASE_DIR / 'output' / 'tables'
if SAVE_OUTPUTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('OUT_DIR:', OUT_DIR)
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

BASE_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터
OUT_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/output/tables
SAVE_OUTPUTS: False


## 3. 데이터 로드 및 변수 확인

최종 분석용 데이터셋은 `working/analysis` 폴더에서만 찾는다. 분석에 필요한 변수 존재 여부와 결측 N을 먼저 확인한다.

In [2]:
patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for path in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': path.name,
            'path': path,
            'modified_time': pd.Timestamp(path.stat().st_mtime, unit='s'),
            'size_bytes': path.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('[working/analysis 데이터 후보]')
if display:
    display(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']])
else:
    print(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']].to_string(index=False))
print('\n사용 데이터 파일:', DATA_PATH.relative_to(BASE_DIR))

suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
required_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry']
variable_check = pd.DataFrame([
    {
        '변수명': var,
        '존재 여부': var in df.columns,
        '유효 N': int(df[var].notna().sum()) if var in df.columns else np.nan,
        '결측 N': int(df[var].isna().sum()) if var in df.columns else np.nan,
    }
    for var in required_vars
])
missing_vars = variable_check.loc[~variable_check['존재 여부'], '변수명'].tolist()

print('전체 N:', f'{total_n:,}')
print('\n[변수 존재 여부 및 결측 확인]')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

if missing_vars:
    raise KeyError('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))

[working/analysis 데이터 후보]


,priority,file,modified_time,size_bytes
0,1,nia_2024_analysis_total.csv,2026-04-27 19:01:15.774730444,1654412



사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv
전체 N: 12,203

[변수 존재 여부 및 결측 확인]


,변수명,존재 여부,유효 N,결측 N
0,effect_proc_improve,True,12203,0
1,it_org_any,True,12203,0
2,ai_use_sum,True,12203,0
3,firm_size,True,12203,0
4,industry,True,12203,0


## 4. 주모형 정의

주모형은 OLS 회귀분석으로 추정한다. 표준오차는 이분산 가능성을 고려해 HC3 robust standard error를 사용한다.

- Model 1: `effect_proc_improve ~ it_org_any + C(firm_size) + C(industry)`
- Model 2: `ai_use_sum ~ it_org_any + C(firm_size) + C(industry)`
- Model 3: `effect_proc_improve ~ it_org_any * ai_use_sum + C(firm_size) + C(industry)`

`firm_size`와 `industry`의 더미 계수는 보고서용 표에서 생략하고, 통제 포함 여부만 표시한다.

In [3]:
models_spec = {
    'Model 1': {
        'title': 'H1 프로세스 개선 효과',
        'hypothesis': 'H1',
        'outcome': 'effect_proc_improve',
        'formula': 'effect_proc_improve ~ it_org_any + C(firm_size) + C(industry)',
        'variables': ['effect_proc_improve', 'it_org_any', 'firm_size', 'industry'],
        'terms': ['it_org_any'],
    },
    'Model 2': {
        'title': 'H2 AI 활용 수준',
        'hypothesis': 'H2',
        'outcome': 'ai_use_sum',
        'formula': 'ai_use_sum ~ it_org_any + C(firm_size) + C(industry)',
        'variables': ['ai_use_sum', 'it_org_any', 'firm_size', 'industry'],
        'terms': ['it_org_any'],
    },
    'Model 3': {
        'title': 'H3 조절효과',
        'hypothesis': 'H3',
        'outcome': 'effect_proc_improve',
        'formula': 'effect_proc_improve ~ it_org_any * ai_use_sum + C(firm_size) + C(industry)',
        'variables': ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry'],
        'terms': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum'],
    },
}

term_labels = {
    'it_org_any': '정보화 담당 체계 보유',
    'ai_use_sum': 'AI 활용 수준',
    'it_org_any:ai_use_sum': '정보화 담당 체계 × AI 활용 수준',
}

model_n_check = []
for model_name, spec in models_spec.items():
    sub = df[spec['variables']].dropna()
    lost_n = total_n - len(sub)
    missing_by_var = {var: int(df[var].isna().sum()) for var in spec['variables']}
    model_n_check.append({
        'model': model_name,
        'formula': spec['formula'],
        '사용 N': len(sub),
        '전체 N 대비 제외 N': lost_n,
        '결측 변수별 N': missing_by_var,
    })
model_n_check_df = pd.DataFrame(model_n_check)
print('[회귀모형별 사용 N 확인]')
if display:
    display(model_n_check_df)
else:
    print(model_n_check_df.to_string(index=False))

[회귀모형별 사용 N 확인]


,model,formula,사용 N,전체 N 대비 제외 N,결측 변수별 N
0,Model 1,effect_proc_improve ~ it_org_any + C(firm_size...,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'f..."
1,Model 2,ai_use_sum ~ it_org_any + C(firm_size) + C(ind...,12203,0,"{'ai_use_sum': 0, 'it_org_any': 0, 'firm_size'..."
2,Model 3,effect_proc_improve ~ it_org_any * ai_use_sum ...,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'a..."


## 5. 주모형 추정 결과

각 모형의 `statsmodels` summary를 참고용으로 출력한다. 보고서에는 아래 summary 전체를 붙이지 않고, 다음 섹션의 보고서용 결과표를 사용한다.

In [4]:
fitted_models = {}
fit_errors = []

for model_name, spec in models_spec.items():
    try:
        model_data = df[spec['variables']].dropna().copy()
        fitted = smf.ols(spec['formula'], data=model_data).fit(cov_type='HC3')
        fitted_models[model_name] = fitted
        print(f'\n===== {model_name}: {spec["title"]} =====')
        print('formula:', spec['formula'])
        print(fitted.summary())
    except Exception as exc:
        fit_errors.append({'model': model_name, 'formula': spec['formula'], 'error': repr(exc)})
        print(f'오류 발생 - {model_name}: {spec["formula"]}')
        print(repr(exc))

if fit_errors:
    raise RuntimeError('일부 주모형 추정 중 오류가 발생했습니다. fit_errors를 확인하세요.')


===== Model 1: H1 프로세스 개선 효과 =====
formula: effect_proc_improve ~ it_org_any + C(firm_size) + C(industry)
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.090
Model:                             OLS   Adj. R-squared:                  0.089
Method:                  Least Squares   F-statistic:                     66.81
Date:                 Sat, 02 May 2026   Prob (F-statistic):          1.53e-244
Time:                         00:40:29   Log-Likelihood:                -13459.
No. Observations:                12203   AIC:                         2.696e+04
Df Residuals:                    12183   BIC:                         2.711e+04
Df Model:                           19                                         
Covariance Type:                   HC3                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------

## 6. 보고서용 결과표

보고서용 표에는 핵심 계수만 포함한다. 계수 셀은 `계수 + 유의표시`와 괄호 안의 HC3 robust standard error로 구성한다.

In [5]:
def stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def coef_cell(result, term):
    if term not in result.params.index:
        return ''
    coef = result.params[term]
    se = result.bse[term]
    p = result.pvalues[term]
    return f'{coef:.3f}{stars(p)}\n({se:.3f})'


def p_display(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '<.001'
    return f'{p:.3f}'


def md_table(data: pd.DataFrame) -> str:
    def fmt(value):
        if pd.isna(value):
            return ''
        if isinstance(value, (float, np.floating)):
            return f'{float(value):.3f}'
        return str(value).replace('|', '\\|').replace('\n', '<br>')
    cols = list(data.columns)
    lines = ['| ' + ' | '.join(cols) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in data.iterrows():
        lines.append('| ' + ' | '.join(fmt(row[col]) for col in cols) + ' |')
    return '\n'.join(lines)


def save_table(data: pd.DataFrame, path: Path, **kwargs):
    if SAVE_OUTPUTS:
        data.to_csv(path, **kwargs)
    return path


def save_text(text: str, path: Path):
    if SAVE_OUTPUTS:
        path.write_text(text, encoding='utf-8')
    return path


def print_save_status(path: Path):
    if SAVE_OUTPUTS:
        print('-', path.relative_to(BASE_DIR))
    else:
        print(f'- {path.name}: 저장 비활성화, 노트북 출력만 확인')

report_rows = []
row_order = [
    ('정보화 담당 체계 보유', 'it_org_any'),
    ('AI 활용 수준', 'ai_use_sum'),
    ('정보화 담당 체계 × AI 활용 수준', 'it_org_any:ai_use_sum'),
]

for label, term in row_order:
    row = {'변수': label}
    for model_name, spec in models_spec.items():
        result = fitted_models[model_name]
        row[spec['title']] = coef_cell(result, term)
    report_rows.append(row)

for control_label in ['기업 규모 통제', '업종 통제']:
    row = {'변수': control_label}
    for model_name, spec in models_spec.items():
        row[spec['title']] = 'Yes'
    report_rows.append(row)

for stat_label, attr in [('N', 'nobs'), ('R-squared', 'rsquared'), ('Adjusted R-squared', 'rsquared_adj')]:
    row = {'변수': stat_label}
    for model_name, spec in models_spec.items():
        result = fitted_models[model_name]
        value = getattr(result, attr)
        row[spec['title']] = f'{int(value):,}' if stat_label == 'N' else f'{value:.3f}'
    report_rows.append(row)

table5 = pd.DataFrame(report_rows)
print('Table 5. 주모형 결과표')
if display:
    display(table5)
else:
    print(table5.to_string(index=False))

# 핵심 계수 tidy table
tidy_rows = []
for model_name, spec in models_spec.items():
    result = fitted_models[model_name]
    conf = result.conf_int(alpha=0.05)
    for term in spec['terms']:
        if term not in result.params.index:
            continue
        tidy_rows.append({
            'model': model_name,
            'hypothesis': spec['hypothesis'],
            'outcome': spec['outcome'],
            'term': term,
            'coef': result.params[term],
            'robust_se': result.bse[term],
            't_value': result.tvalues[term],
            'p_value': result.pvalues[term],
            'significance': stars(result.pvalues[term]),
            'ci_lower': conf.loc[term, 0],
            'ci_upper': conf.loc[term, 1],
            'n': int(result.nobs),
            'r_squared': result.rsquared,
            'adj_r_squared': result.rsquared_adj,
        })

tidy_table = pd.DataFrame(tidy_rows)
round_cols = ['coef', 'robust_se', 't_value', 'p_value', 'ci_lower', 'ci_upper', 'r_squared', 'adj_r_squared']
for col in round_cols:
    tidy_table[col] = tidy_table[col].astype(float).round(3)

print('\n핵심 계수 tidy table')
if display:
    display(tidy_table)
else:
    print(tidy_table.to_string(index=False))

Table 5. 주모형 결과표


,변수,H1 프로세스 개선 효과,H2 AI 활용 수준,H3 조절효과
0,정보화 담당 체계 보유,0.171***\n(0.014),0.105***\n(0.022),0.117***\n(0.016)
1,AI 활용 수준,,,0.008\n(0.009)
2,정보화 담당 체계 × AI 활용 수준,,,0.062***\n(0.011)
3,기업 규모 통제,Yes,Yes,Yes
4,업종 통제,Yes,Yes,Yes
5,N,"12,203","12,203","12,203"
6,R-squared,0.090,0.122,0.097
7,Adjusted R-squared,0.089,0.121,0.096



핵심 계수 tidy table


,model,hypothesis,outcome,term,coef,robust_se,t_value,p_value,significance,ci_lower,ci_upper,n,r_squared,adj_r_squared
0,Model 1,H1,effect_proc_improve,it_org_any,0.171,0.014,11.912,0.000,***,0.143,0.199,12203,0.090,0.089
1,Model 2,H2,ai_use_sum,it_org_any,0.105,0.022,4.699,0.000,***,0.061,0.148,12203,0.122,0.121
2,Model 3,H3,effect_proc_improve,it_org_any,0.117,0.016,7.209,0.000,***,0.085,0.148,12203,0.097,0.096
3,Model 3,H3,effect_proc_improve,ai_use_sum,0.008,0.009,0.848,0.396,,-0.010,0.025,12203,0.097,0.096
4,Model 3,H3,effect_proc_improve,it_org_any:ai_use_sum,0.062,0.011,5.658,0.000,***,0.041,0.084,12203,0.097,0.096


## 7. 주모형 해석 메모

보고서 본문에 붙일 수 있는 3-6문장 해석 메모를 생성한다. 단면자료 기반 예비분석이므로 인과 표현은 피한다.

In [6]:
def get_tidy(model, term):
    return tidy_table.loc[(tidy_table['model'] == model) & (tidy_table['term'] == term)].iloc[0]

m1 = get_tidy('Model 1', 'it_org_any')
m2 = get_tidy('Model 2', 'it_org_any')
m3_int = get_tidy('Model 3', 'it_org_any:ai_use_sum')

h3_direction = '일치한다' if m3_int['coef'] > 0 else '일치하지 않는다'
control_direction = '유지된다' if (m1['coef'] > 0 and m2['coef'] > 0 and m3_int['coef'] > 0) else '일부 약화된다'

interpretation_sentences = [
    f'Model 1에서 정보화 담당 체계 보유 계수는 {m1["coef"]:.3f}{m1["significance"]}로 나타났으며, 이는 기업 규모와 업종을 통제한 후에도 정보화 담당 체계를 보유한 기업의 프로세스 개선 효과가 더 높게 나타난다는 H1의 방향과 일치한다.',
    f'Model 2에서 정보화 담당 체계 보유 계수는 {m2["coef"]:.3f}{m2["significance"]}로 나타나, 정보화 담당 체계 보유 기업의 AI 활용 수준이 더 높게 나타난다는 H2의 방향과 일치한다.',
    f'Model 3에서 정보화 담당 체계 × AI 활용 수준 상호작용항 계수는 {m3_int["coef"]:.3f}{m3_int["significance"]}로, AI 활용 수준이 높을수록 정보화 담당 체계와 프로세스 개선 효과 간 관계가 강화된다는 H3의 방향과 {h3_direction}.',
    f'세 주모형 모두 기업 규모와 업종 더미를 포함했으며, 핵심 계수의 방향은 가설의 예상 방향과 대체로 {control_direction}.',
    '다만 본 분석은 단면자료 기반의 예비 OLS 분석이므로, 결과를 인과관계로 해석하기에는 한계가 있다.',
]

print('주모형 해석 메모')
for sentence in interpretation_sentences:
    print('-', sentence)

주모형 해석 메모
- Model 1에서 정보화 담당 체계 보유 계수는 0.171***로 나타났으며, 이는 기업 규모와 업종을 통제한 후에도 정보화 담당 체계를 보유한 기업의 프로세스 개선 효과가 더 높게 나타난다는 H1의 방향과 일치한다.
- Model 2에서 정보화 담당 체계 보유 계수는 0.105***로 나타나, 정보화 담당 체계 보유 기업의 AI 활용 수준이 더 높게 나타난다는 H2의 방향과 일치한다.
- Model 3에서 정보화 담당 체계 × AI 활용 수준 상호작용항 계수는 0.062***로, AI 활용 수준이 높을수록 정보화 담당 체계와 프로세스 개선 효과 간 관계가 강화된다는 H3의 방향과 일치한다.
- 세 주모형 모두 기업 규모와 업종 더미를 포함했으며, 핵심 계수의 방향은 가설의 예상 방향과 대체로 유지된다.
- 다만 본 분석은 단면자료 기반의 예비 OLS 분석이므로, 결과를 인과관계로 해석하기에는 한계가 있다.


## 8. 확인 필요 사항

결과 저장 여부, 저장 파일 목록, 모형별 사용 N, 결측으로 인한 표본 손실 여부를 확인한다.

In [7]:
# 저장
csv_path = OUT_DIR / 'table5_main_model_results.csv'
md_path = OUT_DIR / 'table5_main_model_results.md'
tidy_path = OUT_DIR / 'table5_main_model_tidy.csv'
interp_path = OUT_DIR / 'main_model_interpretation.md'

save_table(table5, csv_path, index=False, encoding='utf-8-sig')
save_table(tidy_table, tidy_path, index=False, encoding='utf-8-sig')
save_text('# Table 5. 주모형 결과표\n\n' + md_table(table5) + '\n', md_path)
save_text('# 주모형 해석 메모\n\n' + f'- 사용 데이터: `{DATA_PATH.relative_to(BASE_DIR)}`\n- 전체 N: {total_n:,}\n\n' + '\n'.join(f'- {s}' for s in interpretation_sentences) + '\n', interp_path)

need_check = []
if missing_vars:
    need_check.append('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))
else:
    need_check.append('필수 변수 5개가 모두 존재함')

for _, row in model_n_check_df.iterrows():
    if row['전체 N 대비 제외 N'] > 0:
        need_check.append(f"{row['model']}에서 전체 N 대비 {row['전체 N 대비 제외 N']}건 제외됨: {row['결측 변수별 N']}")
if all(row['전체 N 대비 제외 N'] == 0 for _, row in model_n_check_df.iterrows()):
    need_check.append('모든 주모형에서 사용 N이 전체 N과 동일하며, 결측으로 인한 표본 손실 없음')
need_check.append('p-value와 유의표시는 HC3 robust standard error 기준임')
need_check.append('firm_size와 industry 더미 계수는 보고서용 표에서 생략하고 통제 포함 여부만 표시함')

print('1. 사용한 데이터 파일명과 전체 N')
print('-', DATA_PATH.relative_to(BASE_DIR))
print('-', f'N = {total_n:,}')
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

print('\n2. 변수 존재 여부 및 결측 확인표')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

print('\n6. 보고서용 Table 5. 주모형 결과표')
if display:
    display(table5)
else:
    print(table5.to_string(index=False))

print('\n7. 핵심 계수 tidy table')
if display:
    display(tidy_table)
else:
    print(tidy_table.to_string(index=False))

print('\n8. 보고서용 해석 메모')
for sentence in interpretation_sentences:
    print('-', sentence)

print('\n9. 저장 여부 및 저장 파일 목록')
if SAVE_OUTPUTS:
    print('저장 완료')
else:
    print('저장 비활성화: 파일은 생성하지 않고 노트북 화면에만 표시했습니다.')
for path in [csv_path, md_path, tidy_path, interp_path]:
    if SAVE_OUTPUTS:
        print('-', path.relative_to(BASE_DIR))
    else:
        print(f'- {path.name}: 저장 비활성화')

print('\n10. 확인 필요 사항')
for note in need_check:
    print('-', note)

1. 사용한 데이터 파일명과 전체 N
- working/analysis/nia_2024_analysis_total.csv
- N = 12,203
SAVE_OUTPUTS: False

2. 변수 존재 여부 및 결측 확인표


,변수명,존재 여부,유효 N,결측 N
0,effect_proc_improve,True,12203,0
1,it_org_any,True,12203,0
2,ai_use_sum,True,12203,0
3,firm_size,True,12203,0
4,industry,True,12203,0



6. 보고서용 Table 5. 주모형 결과표


,변수,H1 프로세스 개선 효과,H2 AI 활용 수준,H3 조절효과
0,정보화 담당 체계 보유,0.171***\n(0.014),0.105***\n(0.022),0.117***\n(0.016)
1,AI 활용 수준,,,0.008\n(0.009)
2,정보화 담당 체계 × AI 활용 수준,,,0.062***\n(0.011)
3,기업 규모 통제,Yes,Yes,Yes
4,업종 통제,Yes,Yes,Yes
5,N,"12,203","12,203","12,203"
6,R-squared,0.090,0.122,0.097
7,Adjusted R-squared,0.089,0.121,0.096



7. 핵심 계수 tidy table


,model,hypothesis,outcome,term,coef,robust_se,t_value,p_value,significance,ci_lower,ci_upper,n,r_squared,adj_r_squared
0,Model 1,H1,effect_proc_improve,it_org_any,0.171,0.014,11.912,0.000,***,0.143,0.199,12203,0.090,0.089
1,Model 2,H2,ai_use_sum,it_org_any,0.105,0.022,4.699,0.000,***,0.061,0.148,12203,0.122,0.121
2,Model 3,H3,effect_proc_improve,it_org_any,0.117,0.016,7.209,0.000,***,0.085,0.148,12203,0.097,0.096
3,Model 3,H3,effect_proc_improve,ai_use_sum,0.008,0.009,0.848,0.396,,-0.010,0.025,12203,0.097,0.096
4,Model 3,H3,effect_proc_improve,it_org_any:ai_use_sum,0.062,0.011,5.658,0.000,***,0.041,0.084,12203,0.097,0.096



8. 보고서용 해석 메모
- Model 1에서 정보화 담당 체계 보유 계수는 0.171***로 나타났으며, 이는 기업 규모와 업종을 통제한 후에도 정보화 담당 체계를 보유한 기업의 프로세스 개선 효과가 더 높게 나타난다는 H1의 방향과 일치한다.
- Model 2에서 정보화 담당 체계 보유 계수는 0.105***로 나타나, 정보화 담당 체계 보유 기업의 AI 활용 수준이 더 높게 나타난다는 H2의 방향과 일치한다.
- Model 3에서 정보화 담당 체계 × AI 활용 수준 상호작용항 계수는 0.062***로, AI 활용 수준이 높을수록 정보화 담당 체계와 프로세스 개선 효과 간 관계가 강화된다는 H3의 방향과 일치한다.
- 세 주모형 모두 기업 규모와 업종 더미를 포함했으며, 핵심 계수의 방향은 가설의 예상 방향과 대체로 유지된다.
- 다만 본 분석은 단면자료 기반의 예비 OLS 분석이므로, 결과를 인과관계로 해석하기에는 한계가 있다.

9. 저장 여부 및 저장 파일 목록
저장 비활성화: 파일은 생성하지 않고 노트북 화면에만 표시했습니다.
- table5_main_model_results.csv: 저장 비활성화
- table5_main_model_results.md: 저장 비활성화
- table5_main_model_tidy.csv: 저장 비활성화
- main_model_interpretation.md: 저장 비활성화

10. 확인 필요 사항
- 필수 변수 5개가 모두 존재함
- 모든 주모형에서 사용 N이 전체 N과 동일하며, 결측으로 인한 표본 손실 없음
- p-value와 유의표시는 HC3 robust standard error 기준임
- firm_size와 industry 더미 계수는 보고서용 표에서 생략하고 통제 포함 여부만 표시함
